# 05 · Álbum de mapas interactivos

| | |
|---|---|
| **Objetivo** | Integrar en mapas web navegables los resultados de los notebooks 02, 03, 07 y 08, para exploración y comunicación a usuarios de negocio. |
| **Herramienta** | folium (Leaflet), capas conmutables, ventanas emergentes por distrito, agrupación de marcadores y mapa de calor. |
| **Salidas** | `outputs/mapas/*.html` (se abren en cualquier navegador, sin servidor). |

Los mapas no se incrustan en el notebook para que el archivo se mantenga liviano y GitHub pueda mostrarlo;
cada celda guarda un HTML y devuelve su ruta.

In [1]:
from pathlib import Path
import json

import branca.colormap as cm
import folium
import geopandas as gpd
import numpy as np
import pandas as pd
from folium.plugins import Fullscreen, HeatMap, MarkerCluster, MiniMap

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed"
MAPAS = ROOT / "outputs" / "mapas"
MAPAS.mkdir(parents=True, exist_ok=True)
CENTRO = [-6.6, -79.4]
TILES = "OpenStreetMap"   # capa base sin clave de API

## 1. Datos integrados

In [2]:
gpkg = DATA / "geobanca_norte.gpkg"
d = gpd.read_file(gpkg, layer="distritos")
d = (d.merge(pd.read_csv(DATA / "base_aede.csv", dtype={"ubigeo": str})[["ubigeo", "acceso_eb"]], on="ubigeo", how="left")
      .merge(pd.read_csv(DATA / "lisa_acceso.csv", dtype={"ubigeo": str})[["ubigeo", "lisa", "lisa_p", "gi_clase", "lisa_bv"]], on="ubigeo", how="left")
      .merge(pd.read_csv(DATA / "brecha_acceso_2025.csv", dtype={"ubigeo": str})[["ubigeo", "tasa_esperada", "brecha"]], on="ubigeo", how="left"))

# Geometría simplificada (~150 m) y coordenadas redondeadas a 1e-4 grados (~11 m) para que los HTML pesen poco
import shapely
d = d.to_crs("EPSG:32717")
d["geometry"] = d.geometry.simplify(150, preserve_topology=True)
d = d.to_crs("EPSG:4326")
d["geometry"] = shapely.set_precision(d.geometry.values, 1e-4)
deps = d.dissolve("departamento").reset_index()[["departamento", "geometry"]]

sitios = gpd.read_file(gpkg, layer="osm_bancos_atm").to_crs("EPSG:4326")
estaciones = gpd.read_file(gpkg, layer="senamhi_estaciones").to_crs("EPSG:4326")

for c in ["acceso_eb", "pobreza_2018", "pct_urbana", "tasa_esperada", "brecha", "creditos_pc_soles"]:
    d[c] = d[c].astype(float).round(2)
d["pob_total_2025"] = d["pob_total_2025"].fillna(0).astype(int)
print(f"{len(d)} distritos | {len(sitios)} sitios OSM | {len(estaciones)} estaciones SENAMHI")

327 distritos | 395 sitios OSM | 175 estaciones SENAMHI


In [3]:
CAMPOS = ["distrito", "provincia", "departamento", "pob_total_2025", "acceso_eb", "pobreza_2018",
          "pct_urbana", "lisa", "brecha"]
ALIAS = ["Distrito", "Provincia", "Departamento", "Población 2025", "Acceso (EB, por 10 mil adultos)",
         "Pobreza 2018 (%)", "Población urbana (%)", "Cluster LISA", "Brecha (log obs − log esp)"]

def capa_coropletica(m, columna, colormap, nombre, mostrar=True):
    colormap.caption = nombre
    fg = folium.FeatureGroup(name=nombre, show=mostrar)
    folium.GeoJson(
        d[CAMPOS + ["geometry"]].assign(_v=d[columna]).to_json(),
        style_function=lambda f: {"fillColor": colormap(f["properties"]["_v"]) if f["properties"]["_v"] is not None else "#cccccc",
                                  "color": "white", "weight": 0.3, "fillOpacity": 0.8},
        highlight_function=lambda f: {"weight": 2, "color": "black"},
        tooltip=folium.GeoJsonTooltip(fields=CAMPOS, aliases=ALIAS, localize=True, sticky=True),
    ).add_to(fg)
    fg.add_to(m)
    return colormap

def capa_categorica(m, columna, colores, nombre, mostrar=False):
    fg = folium.FeatureGroup(name=nombre, show=mostrar)
    folium.GeoJson(
        d[CAMPOS + ["geometry"]].assign(_c=d[columna]).to_json(),
        style_function=lambda f: {"fillColor": colores.get(f["properties"]["_c"], "#e6e6e6"), "color": "white",
                                  "weight": 0.3, "fillOpacity": 0.85 if f["properties"]["_c"] in colores else 0.25},
        tooltip=folium.GeoJsonTooltip(fields=CAMPOS, aliases=ALIAS, sticky=True),
    ).add_to(fg)
    fg.add_to(m)

def leyenda_html(titulo, colores):
    filas = "".join(f'<div><span style="background:{c};width:12px;height:12px;display:inline-block;margin-right:6px"></span>{k}</div>'
                    for k, c in colores.items())
    estilo = ("position:fixed;bottom:30px;left:10px;z-index:9999;background:white;padding:8px 10px;"
              "border-radius:6px;box-shadow:0 1px 4px rgba(0,0,0,.3);font-size:12px")
    return folium.Element(f'<div style="{estilo}"><b>{titulo}</b>{filas}</div>')

def base(zoom=7):
    m = folium.Map(location=CENTRO, zoom_start=zoom, tiles=TILES, control_scale=True)
    folium.GeoJson(deps.to_json(), name="Departamentos",
                   style_function=lambda f: {"fill": False, "color": "#333333", "weight": 1.2}).add_to(m)
    Fullscreen().add_to(m)
    MiniMap(toggle_display=True).add_to(m)
    return m

## 2. Mapa principal: acceso, pobreza, clusters y brecha

In [4]:
m = base()
cmap_acc = cm.LinearColormap(["#ffffcc", "#41b6c4", "#081d58"], vmin=float(d["acceso_eb"].quantile(0.02)),
                             vmax=float(d["acceso_eb"].quantile(0.98)))
capa_coropletica(m, "acceso_eb", cmap_acc, "Acceso a canales (por 10 mil adultos, EB)").add_to(m)
cmap_pob = cm.LinearColormap(["#fff5eb", "#fd8d3c", "#7f2704"], vmin=0, vmax=80)
capa_coropletica(m, "pobreza_2018", cmap_pob, "Pobreza monetaria 2018 (%)", mostrar=False)

COL_LISA = {"Alto-Alto": "#d7191c", "Bajo-Bajo": "#2c7bb6", "Bajo-Alto": "#abd9e9", "Alto-Bajo": "#fdae61"}
capa_categorica(m, "lisa", COL_LISA, "Clusters LISA (p < 0,05)")
COL_BV = {"Pobre / vecinos sin acceso": "#7b3294", "No pobre / vecinos con acceso": "#008837",
          "Pobre / vecinos con acceso": "#c2a5cf", "No pobre / vecinos sin acceso": "#a6dba0"}
capa_categorica(m, "lisa_bv", COL_BV, "LISA bivariado pobreza / acceso de vecinos")

lim = float(np.nanpercentile(np.abs(d["brecha"]), 97))
cmap_br = cm.LinearColormap(["#b2182b", "#f7f7f7", "#2166ac"], vmin=-lim, vmax=lim)
capa_coropletica(m, "brecha", cmap_br, "Brecha de acceso (rojo = desatendido)", mostrar=False)

folium.LayerControl(collapsed=False).add_to(m)
m.get_root().html.add_child(leyenda_html("Clusters LISA", {**COL_LISA, "No significativo": "#e6e6e6"}))
ruta = MAPAS / "05_mapa_principal.html"
m.save(ruta)
print(ruta.relative_to(ROOT), f"{ruta.stat().st_size / 1e6:.1f} MB")

outputs/mapas/05_mapa_principal.html 3.2 MB


## 3. Desiertos financieros: distritos Bajo-Bajo y su población

In [5]:
m = base(zoom=8)
des = d[d["lisa"] == "Bajo-Bajo"]
folium.GeoJson(d[["geometry"]].to_json(), style_function=lambda f: {"fillColor": "#f2f2f2", "color": "white",
               "weight": 0.3, "fillOpacity": 0.5}).add_to(m)
folium.GeoJson(
    des[CAMPOS + ["geometry"]].to_json(), name="Desiertos financieros",
    style_function=lambda f: {"fillColor": "#2c7bb6", "color": "#08306b", "weight": 1, "fillOpacity": 0.75},
    tooltip=folium.GeoJsonTooltip(fields=CAMPOS, aliases=ALIAS, sticky=True)).add_to(m)
for _, r in des.iterrows():
    p = r.geometry.representative_point()
    folium.CircleMarker([p.y, p.x], radius=3 + np.sqrt(r["pob_total_2025"]) / 25, color="#08306b",
                        fill=True, fill_opacity=0.9, weight=1,
                        popup=f"<b>{r['distrito']}</b><br>{r['pob_total_2025']:,} hab.<br>acceso EB: {r['acceso_eb']}").add_to(m)
m.fit_bounds(des.total_bounds[[1, 0, 3, 2]].reshape(2, 2).tolist())
ruta = MAPAS / "05_desiertos_financieros.html"
m.save(ruta)
print(ruta.relative_to(ROOT), f"{ruta.stat().st_size / 1e6:.1f} MB | {len(des)} distritos, {des['pob_total_2025'].sum():,} habitantes")

outputs/mapas/05_desiertos_financieros.html 0.8 MB | 19 distritos, 212,000 habitantes


## 4. Oferta georreferenciada: bancos y cajeros (OSM) con mapa de calor

In [6]:
m = base()
ICONO = {"bank": ("university", "darkblue"), "atm": ("credit-card", "green"),
         "bureau_de_change": ("usd", "orange"), "payment_centre": ("shopping-cart", "gray")}
cl = MarkerCluster(name="Bancos y ATM (OSM)").add_to(m)
for _, r in sitios.iterrows():
    ic, col = ICONO.get(r["amenity"], ("info-sign", "gray"))
    nombre = r["nombre"] if isinstance(r["nombre"], str) else (r["marca"] if isinstance(r["marca"], str) else r["amenity"])
    folium.Marker([r.geometry.y, r.geometry.x], popup=f"<b>{nombre}</b><br>{r['amenity']}",
                  icon=folium.Icon(icon=ic, prefix="fa", color=col)).add_to(cl)
HeatMap(np.column_stack([sitios.geometry.y, sitios.geometry.x]).tolist(), name="Mapa de calor",
        radius=14, blur=18, min_opacity=0.3, show=False).add_to(m)
fg_est = folium.FeatureGroup(name="Estaciones SENAMHI", show=False)
for _, r in estaciones.iterrows():
    folium.CircleMarker([r.geometry.y, r.geometry.x], radius=3, color="#6a3d9a", fill=True, fill_opacity=0.8,
                        popup=f"{r['nombre']} ({r['categoria']}, {r['estado']})").add_to(fg_est)
fg_est.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
ruta = MAPAS / "05_oferta_osm.html"
m.save(ruta)
print(ruta.relative_to(ROOT), f"{ruta.stat().st_size / 1e6:.1f} MB")

outputs/mapas/05_oferta_osm.html 0.9 MB


## 5. Índice de mapas generados

In [7]:
for f in sorted(MAPAS.glob("05_*.html")):
    print(f"{f.name:32s} {f.stat().st_size / 1e6:5.1f} MB")

05_desiertos_financieros.html      0.8 MB
05_mapa_principal.html             3.2 MB
05_oferta_osm.html                 0.9 MB


**Cómo verlos.** Abrir los archivos de `outputs/mapas/` con doble clic. Para publicarlos en la web basta activar
GitHub Pages en el repositorio y enlazar los HTML desde el README.

**Qué muestra cada mapa**

| Archivo | Contenido | Uso |
|---|---|---|
| `05_mapa_principal.html` | Acceso suavizado, pobreza, clusters LISA, LISA bivariado y brecha, como capas conmutables con ficha por distrito | Exploración general y exposición |
| `05_desiertos_financieros.html` | Los 19 distritos Bajo-Bajo con círculos proporcionales a su población | Priorización comercial |
| `05_oferta_osm.html` | Bancos y ATM agrupados, mapa de calor y estaciones SENAMHI | Contexto de la oferta y de la red de monitoreo climático |